In [1]:
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup

# NLTK
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download NLTK resources (only first time)
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Error loading stopwords: <urlopen error pathsec.urlopen:
[nltk_data]     no validated address for host
[nltk_data]     'raw.githubusercontent.com'; refusing to connect by
[nltk_data]     unvalidated hostname>
[nltk_data] Error loading wordnet: <urlopen error pathsec.urlopen: no
[nltk_data]     validated address for host
[nltk_data]     'raw.githubusercontent.com'; refusing to connect by
[nltk_data]     unvalidated hostname>
[nltk_data] Error loading omw-1.4: <urlopen error pathsec.urlopen: no
[nltk_data]     validated address for host
[nltk_data]     'raw.githubusercontent.com'; refusing to connect by
[nltk_data]     unvalidated hostname>


False

In [2]:
df = pd.read_csv("../data/raw/fake_job_postings.csv")

print(df.shape)
df.head()

(17880, 18)


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [3]:
df.drop(columns=["job_id", "salary_range", "department"], inplace=True)

print(df.shape)

(17880, 15)


In [4]:
df.dropna(subset=["description"], inplace=True)

In [5]:
text_columns = [
    "company_profile",
    "requirements",
    "benefits"
]

for col in text_columns:
    df[col] = df[col].fillna("")

In [6]:
categorical_columns = [
    "location",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

for col in categorical_columns:
    df[col] = df[col].fillna("Unknown")

In [7]:
df.isnull().sum()

title                  0
location               0
company_profile        0
description            0
requirements           0
benefits               0
telecommuting          0
has_company_logo       0
has_questions          0
employment_type        0
required_experience    0
required_education     0
industry               0
function               0
fraudulent             0
dtype: int64

In [8]:
df["text"] = (
    df["title"] + " " +
    df["company_profile"] + " " +
    df["description"] + " " +
    df["requirements"] + " " +
    df["benefits"]
)

In [9]:
df["text"].iloc[0]

"Marketing Intern We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connect, and celebrate home cooks, and give them everything they need in one place.We have a top editorial, business, and engineering team. We're focused on using technology to find new and better ways to connect people around their specific food interests, and to offer them superb, highly curated information about food and cooking. We attract the most talented home cooks and contributors in the country; we also publish well-known professionals like Mario Batali, Gwyneth Paltrow, and Danny Meyer. And we have partnerships with Whole Foods Market and Random House.Food52 has been named the best food website by the James Beard Foundation and IACP, and has been featured in the New York Times, NPR, Pando Daily, TechCrunch, and on the Today Show.We're located in Chelsea, in New York City. Food52, a fast-growing, James Beard Award-winning online food community and crowd-sourced and cu

In [10]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [11]:
stop_words = ENGLISH_STOP_WORDS

In [12]:
import re
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = ENGLISH_STOP_WORDS

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # Remove HTML
    text = BeautifulSoup(text, "html.parser").get_text()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Remove stopwords
    words = [
        word for word in text.split()
        if word not in stop_words
    ]

    return " ".join(words)

In [13]:
df["clean_text"] = df["text"].apply(clean_text)

In [14]:
print("Original:\n")
print(df["text"].iloc[0])

print("\n" + "="*80 + "\n")

print("Cleaned:\n")
print(df["clean_text"].iloc[0])

Original:

Marketing Intern We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connect, and celebrate home cooks, and give them everything they need in one place.We have a top editorial, business, and engineering team. We're focused on using technology to find new and better ways to connect people around their specific food interests, and to offer them superb, highly curated information about food and cooking. We attract the most talented home cooks and contributors in the country; we also publish well-known professionals like Mario Batali, Gwyneth Paltrow, and Danny Meyer. And we have partnerships with Whole Foods Market and Random House.Food52 has been named the best food website by the James Beard Foundation and IACP, and has been featured in the New York Times, NPR, Pando Daily, TechCrunch, and on the Today Show.We're located in Chelsea, in New York City. Food52, a fast-growing, James Beard Award-winning online food community and crowd-sour

In [15]:
model_df = df[
    [
        "clean_text",
        "telecommuting",
        "has_company_logo",
        "has_questions",
        "fraudulent"
    ]
]

model_df.head()

,clean_text,telecommuting,has_company_logo,has_questions,fraudulent
0,marketing intern food weve created groundbreak...,0,1,0,0
1,customer service cloud video production second...,0,1,0,0
2,commissioning machinery assistant cma valor se...,0,1,0,0
3,account executive washington dc passion improv...,0,1,0,0
4,review manager spotsource solutions llc global...,0,1,1,0


In [16]:
model_df.to_csv(
    "../data/processed/processed_jobs.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!


In [17]:
print(df["clean_text"].iloc[0][:500])

marketing intern food weve created groundbreaking awardwinning cooking site support connect celebrate home cooks need placewe editorial business engineering team focused using technology new better ways connect people specific food interests offer superb highly curated information food cooking attract talented home cooks contributors country publish wellknown professionals like mario batali gwyneth paltrow danny meyer partnerships foods market random housefood named best food website james beard


In [19]:
import os

print(os.getcwd())

/Users/gopalkumarrajwar/Fake_Job_Detection/notebooks


In [20]:
import os

print(os.path.exists("data/raw/fake_job_postings.csv"))


False


In [22]:
from pathlib import Path

for file in Path(".").rglob("fake_job_postings.csv"):
    print(file)

In [23]:
import os

print(os.listdir(".."))

['.DS_Store', 'app', 'LICENSE', 'requirements.txt', 'models', 'README.md', '.gitignore', '.venv', '.git', '.vscode', 'data', 'assets', 'notebooks', 'reports', 'src']


In [24]:
print(os.listdir("../data"))

['.DS_Store', 'processed', 'raw']


In [25]:
print(os.listdir("../data/raw"))

['.DS_Store', 'fake_job_postings.csv']


In [26]:
import pandas as pd

df_original = pd.read_csv("../data/raw/fake_job_postings.csv")

In [27]:
print(df_original.columns.tolist())

['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']


In [28]:
fake_jobs = df_original[df_original["fraudulent"] == 1]

sample = fake_jobs.sample(1, random_state=42)

sample

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
17789,17790,customer service rep,"US, CA, sacremento",NaN,NaN,NaN,customer service reps needed asap,NaN,will explain on phone interview,0,0,1,NaN,NaN,NaN,NaN,NaN,1


In [29]:
sample[[
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits",
    "telecommuting",
    "has_company_logo",
    "has_questions"
]].T

,17789
title,customer service rep
company_profile,NaN
description,customer service reps needed asap
requirements,NaN
benefits,will explain on phone interview
telecommuting,0
has_company_logo,0
has_questions,1
